In [ ]:
import copy
from typing import Callable
import random

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import OrdinalEncoder
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from logic.recommender_dl import move_tensors, train_val_split
from logic.two_towers import UserTower, MovieTower, TwoTowerModel
from logic.recall_at_k import recall_at_k_macro
from logic.retriever_loss import info_nce, build_interaction_matrix

In [ ]:
device_name = "cpu"

In [ ]:
def _run_retrieval_epoch(
    model: TwoTowerModel,
    dataloader: DataLoader[dict[str, torch.Tensor]],
    var_key_map: dict[str, str],
    device: torch.device,
    is_train: bool,
    embedding_loss: Callable,
    optimizer: optim.Optimizer | None = None,
    **loss_kwargs,
) -> float:
    if is_train:
        model.train()
    else:
        model.eval()
    
    epoch_loss = 0.0
    dataset_len = 0
    
    with torch.set_grad_enabled(is_train):
        for batch in dataloader:
            tensors = move_tensors(var_key_map, batch, device)

            user_vectors, movie_vectors = model(**tensors)
            
            loss = embedding_loss(
                user_vectors,
                movie_vectors,
                tensors["user_ids"],
                tensors["movie_ids"],
                **loss_kwargs,
            )
            
            if is_train and optimizer is not None:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                
            batch_size = user_vectors.size(0)
            epoch_loss += loss.item() * batch_size
            dataset_len += batch_size
            
    return epoch_loss / dataset_len


def train_retriever(
    model: TwoTowerModel,
    train_loader: DataLoader,
    val_loader: DataLoader,
    all_movie_ids: torch.Tensor,
    all_genres: torch.Tensor,
    cold_user_ids: torch.Tensor,
    var_key_map: dict[str, str],
    embedding_loss: Callable,
    epochs=10_000,
    learning_rate=0.001,
    patience=10,
    tol=1e-4,
    weight_decay=1e-4,
    device_name="cpu",
    recall_k=100,
    **loss_kwargs,
) -> None:
    
    device = torch.device(device_name)
    model = model.to(device)
    
    optimizer = optim.AdamW(
        model.parameters(), 
        lr=learning_rate,
        weight_decay=weight_decay,
    )
    
    best_val_recall = -1
    best_val_loss = np.inf
    no_improve_count = 0
    best_weights = None
    
    for epoch in range(epochs):
        avg_train_loss = _run_retrieval_epoch(
            model=model,
            dataloader=train_loader,
            var_key_map=var_key_map,
            device=device,
            is_train=True,
            embedding_loss=embedding_loss,
            optimizer=optimizer,
            **loss_kwargs,
        )
        
        avg_val_loss = _run_retrieval_epoch(
            model=model,
            dataloader=val_loader,
            var_key_map=var_key_map,
            device=device,
            is_train=False,
            embedding_loss=embedding_loss,
            **loss_kwargs,
        )

        val_recall = recall_at_k_macro(
            two_towers=model,
            dataloader=val_loader,
            var_key_map=var_key_map,
            all_movie_ids=all_movie_ids,
            all_genres=all_genres,
            device=device,
            cold_user_ids=cold_user_ids,
            k=recall_k,
        )["global"]
        
        print(
            f"Epoch {epoch+1}/{epochs}, "
            f"Train Loss: {avg_train_loss:.4f}, "
            f"Val Loss: {avg_val_loss:.4f}. ",
            f"Val recall@{recall_k}-macro: {val_recall:.4f}"
        )
        
        if val_recall > best_val_recall:
            best_val_recall = val_recall
            best_weights = copy.deepcopy(
                model.state_dict(),
            )

        if avg_val_loss < (best_val_loss - tol):
            best_val_loss = avg_val_loss
            no_improve_count = 0
        else:
            no_improve_count += 1
            
        if no_improve_count >= patience:
            break
    
    if best_weights is not None:
        model.load_state_dict(best_weights)
        print(
            "Best weights restored. "
            f"Validation recall@{recall_k}-micro is {best_val_recall:.4f}",
        )


In [6]:
with duckdb.connect("/home/maria-khanina/mkgit/movies_project/data/database.duckdb", read_only=True) as db_conn:
    ratings_df = db_conn.execute("""
        SELECT *
        FROM ratings
            INNER JOIN movies USING (movie_id)
            INNER JOIN users USING (user_id)
    """).df()

In [7]:
genres_encoded = ratings_df["genres"].str.get_dummies(sep="|")
genres_cols = genres_encoded.columns.to_list()
genres_num = len(genres_cols)
ratings_df = pd.concat([ratings_df, genres_encoded], axis=1)

In [8]:
encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1,
)

cat_cols = ["gender"]

ratings_df[cat_cols] = encoder.fit_transform(ratings_df[cat_cols]) + 1
ratings_df["occupation"] = ratings_df["occupation"] + 1

In [9]:
max_user = int(ratings_df["user_id"].max())
max_movie = int(ratings_df["movie_id"].max())
max_age = int(ratings_df["age"].max())
max_gender = int(ratings_df["gender"].max())
max_job = int(ratings_df["occupation"].max())

In [10]:
cold_user_ratio = 0.05

In [11]:
train_all_df, test_df = train_val_split(
    ratings_df,
    user_col="user_id",
    time_col="timestamp",
    cold_user_ratio=cold_user_ratio,
    max_test_ratio=0.1,
    test_max_per_var=8,
    test_min_per_var=3,
)

train_big_df, outer_val_df = train_val_split(
    train_all_df,
    user_col="user_id",
    time_col="timestamp",
    cold_user_ratio=cold_user_ratio,
    max_test_ratio=0.1,
    test_max_per_var=8,
    test_min_per_var=3,
)

train_df, val_df = train_val_split(
    train_big_df,
    user_col="user_id",
    time_col="timestamp",
    cold_user_ratio=cold_user_ratio,
    max_test_ratio=0.1,
    test_max_per_var=8,
    test_min_per_var=3,
)

In [ ]:
movie_counts = train_df["movie_id"].value_counts().sort_index()

popularity = torch.ones(
    max_movie + 1, 
    dtype=torch.float32,
)

popularity[movie_counts.index.values] += torch.tensor(
    movie_counts.values, 
    dtype=torch.float32,
)

frequencies = popularity / popularity.sum()
movie_log_freqs = torch.log(frequencies)

device = torch.device(device_name)
movie_log_freqs = movie_log_freqs.to(device)

In [13]:
var_key_map = {
    "user_ids": "user_id",
    "movie_ids": "movie_id",
    "ages": "age",
    "genders": "gender",
    "jobs": "job",
    "genres": "genre",
}

In [14]:
class PositiveRatings(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        genres_cols: list[str],
        min_rating=4.0,
        is_training=False,
        user_dropout=0.0,
        trained_users: torch.Tensor | None = None,
    ) -> None:
        super().__init__()
        
        positive_df = df[df["rating"] >= min_rating].reset_index(
            drop=True,
        )
        

        self.real_user_ids = torch.tensor(
            positive_df["user_id"].values,
            dtype=torch.long,
        )

        self.user_ids = self.real_user_ids.clone()
        if not is_training:
            new_users_mask = ~torch.isin(self.user_ids, trained_users)
            self.user_ids[new_users_mask] = 0
    
        self.movie_ids = torch.tensor(
            positive_df["movie_id"].values, 
            dtype=torch.long,
        )
        self.ages = torch.tensor(
            positive_df["age"].values, 
            dtype=torch.long,
        )
        self.genders = torch.tensor(
            positive_df["gender"].values, 
            dtype=torch.long,
        )
        self.jobs = torch.tensor(
            positive_df["occupation"].values, 
            dtype=torch.long,
        )
        self.genres = torch.tensor(
            positive_df[genres_cols].values, 
            dtype=torch.float32,
        )

        self.user_dropout = user_dropout
        self.is_training = is_training

    def __len__(self) -> int:
        return len(self.user_ids)

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        user_id = self.user_ids[idx]
        if self.is_training and random.random() < self.user_dropout:
            user_id = torch.tensor(0, dtype=torch.long)

        return {
            "user_id": user_id,
            "real_user_id": self.real_user_ids[idx],
            "movie_id": self.movie_ids[idx],
            "age": self.ages[idx],
            "gender": self.genders[idx],
            "job": self.jobs[idx],
            "genre": self.genres[idx],
        }

In [15]:
train_dataset = PositiveRatings(
    train_df, 
    genres_cols,
    min_rating=4,
    is_training=True,
    user_dropout=cold_user_ratio,
)

trained_users = torch.unique(train_dataset.user_ids)

val_dataset = PositiveRatings(
    val_df, 
    genres_cols, 
    trained_users=trained_users,
)
outer_val_dataset = PositiveRatings(
    outer_val_df, 
    genres_cols,
    trained_users=trained_users,
)

train_loader = DataLoader(
    train_dataset, 
    batch_size=2048,
    shuffle=True,
    num_workers=4,
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=2048,
    shuffle=False,
    num_workers=4,
)

outer_val_loader = DataLoader(
    outer_val_dataset, 
    batch_size=2048,
    shuffle=False,
    num_workers=4,
)

In [16]:
train_users = train_df["user_id"].unique()
outer_val_users = outer_val_df["user_id"].unique()
val_users = val_df["user_id"].unique()

outer_cold_user_ids = torch.tensor(
    np.setdiff1d(outer_val_users, train_users),
    dtype=torch.long,
)

cold_user_ids = torch.tensor(
    np.setdiff1d(val_users, train_users),
    dtype=torch.long,
)

In [17]:
interactions = build_interaction_matrix(
    user_indices=train_dataset.user_ids,
    movie_indices=train_dataset.movie_ids,
    max_user_id=max_user,
    max_movie_id=max_movie,
)

In [18]:
def get_unique_movies(
    df: pd.DataFrame,
    genres_cols: list[str],
) -> tuple[torch.Tensor, torch.Tensor]:
    
    unique_movies_df = df.drop_duplicates(
        subset=["movie_id"],
    ).sort_values(
        by="movie_id",
    ).reset_index(
        drop=True,
    )
    
    movie_ids = torch.tensor(
        unique_movies_df["movie_id"].values,
        dtype=torch.long,
    )
    
    movie_genres = torch.tensor(
        unique_movies_df[genres_cols].values,
        dtype=torch.float32,
    )
    
    return movie_ids, movie_genres

unique_movie_ids, unique_movie_genres = get_unique_movies(
    ratings_df,
    genres_cols,
)

In [19]:
user_tower = UserTower(
    users_num=max_user,
    ages_num=max_age,
    genders_num=max_gender,
    jobs_num=max_job,
    user_dim=32,
    age_dim=4,
    gender_dim=2,
    job_dim=8,
    output_dim=64,
    hidden_dims=[256, 128],
    dropout_ratio=0.1,
)
    
movie_tower = MovieTower(
    movies_num=max_movie,
    genres_num=genres_num,
    movie_dim=32,
    genre_dim=8,
    output_dim=64,
    hidden_dims=[256, 128],
    dropout_ratio=0.1,
)

retriever = TwoTowerModel(
    user_tower=user_tower,
    movie_tower=movie_tower,
    normalize_outputs=True,
)

In [20]:
train_retriever(
    model=retriever,
    train_loader=train_loader,
    val_loader=val_loader,
    all_movie_ids=unique_movie_ids,
    all_genres=unique_movie_genres,
    cold_user_ids=cold_user_ids,
    var_key_map=var_key_map,
    embedding_loss=info_nce,
    epochs=10_000,
    learning_rate=0.002,
    patience=30,
    tol=0.0001,
    weight_decay=0.0001,
    device_name=device_name,
    recall_k=100,
    interaction_matrix=interactions,
    check_interactions=True,
    temperature=0.05,
    movie_log_freqs=movie_log_freqs,
)

/home/maria-khanina/mkgit/venv/lib/python3.12/site-packages/torch/autograd/graph.py:865: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch 1/10000, Train Loss: 7.9106, Val Loss: 7.7579.  Val recall@100-macro: 0.2107
Epoch 2/10000, Train Loss: 7.4086, Val Loss: 7.6171.  Val recall@100-macro: 0.2205
Epoch 3/10000, Train Loss: 7.2546, Val Loss: 7.5485.  Val recall@100-macro: 0.2324
Epoch 4/10000, Train Loss: 7.1328, Val Loss: 7.5113.  Val recall@100-macro: 0.2468
Epoch 5/10000, Train Loss: 7.0375, Val Loss: 7.4792.  Val recall@100-macro: 0.2548
Epoch 6/10000, Train Loss: 6.9584, Val Loss: 7.4547.  Val recall@100-macro: 0.2699
Epoch 7/10000, Train Loss: 6.8841, Val Loss: 7.4354.  Val recall@100-macro: 0.2811
Epoch 8/10000, Train Loss: 6.8229, Val Loss: 7.4225.  Val recall@100-macro: 0.2875
Epoch 9/10000, Train Loss: 6.7742, Val Loss: 7.4169.  Val recall@100-macro: 0.2951
Epoch 10/10000, Train Loss: 6.7316, Val Loss: 7.4002.  Val recall@100-macro: 0.3043
Epoch 11/10000, Train Loss: 6.6961, Val Loss: 7.3890.  Val recall@100-macro: 0.3027
Epoch 12/10000, Train Loss: 6.6682, Val Loss: 7.3909.  Val recall@100-macro: 0.3054
E

In [21]:
for k in [10, 50, 100, 200, 500]:
    val_recall = recall_at_k_macro(
        retriever,
        outer_val_loader,
        var_key_map,
        unique_movie_ids,
        unique_movie_genres,
        torch.device("cpu"),
        outer_cold_user_ids,
        k=k,
    )
    print(f"Outer validation recall@{k}: {val_recall}")

Outer validation recall@10: {'global': 0.027697809040546417, 'warm': 0.02607308141887188, 'cold': 0.042495135217905045}
Outer validation recall@50: {'global': 0.16216729581356049, 'warm': 0.1614694744348526, 'cold': 0.1685228794813156}
Outer validation recall@100: {'global': 0.2836320996284485, 'warm': 0.28600752353668213, 'cold': 0.2619975209236145}
Outer validation recall@200: {'global': 0.44994524121284485, 'warm': 0.45498237013816833, 'cold': 0.4040693938732147}
Outer validation recall@500: {'global': 0.7016550302505493, 'warm': 0.7092399597167969, 'cold': 0.6325755715370178}
